In [ ]:
現代地圖切圖例

In [4]:
# ===== 從 eval_data_perfomer 的 json 內 poly 框，切出每張圖的唯一 legend（同名 _poly 不重複切） =====
from pathlib import Path
import json
import shutil
from collections import Counter
import cv2
import numpy as np

# =========================
# 0) 路徑
# =========================
SRC_DIR = Path("/data/ch21908234/work/SOTA_DATA/AI4CMA_evaluation_block/_poly_eq_2/eval_data_perfomer")
DST_ROOT = Path("/data/ch21908234/work/SAM_Classifier_Feedback_Loop/data/modern/default_Legend")

DST_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# 1) 工具函式
# =========================
IMG_EXTS = [".tif", ".tiff", ".png", ".jpg", ".jpeg"]

def find_image_for_json(json_path: Path, meta: dict) -> Path | None:
    """
    優先順序：
    1. json 同層 + imagePath
    2. SRC_DIR + imagePath
    3. json 同 stem，在 json 同層找
    4. json 同 stem，在 SRC_DIR 遞迴找
    """
    image_path_in_json = meta.get("imagePath", None)

    # 1) json 同層 + imagePath
    if image_path_in_json:
        p = json_path.parent / image_path_in_json
        if p.exists():
            return p

        # 2) SRC_DIR + imagePath
        p = SRC_DIR / image_path_in_json
        if p.exists():
            return p

    # 3) json 同 stem，在 json 同層找
    stem = json_path.stem
    for ext in IMG_EXTS:
        p = json_path.parent / f"{stem}{ext}"
        if p.exists():
            return p

    # 4) json 同 stem，在 SRC_DIR 遞迴找
    candidates = []
    for ext in IMG_EXTS:
        candidates.extend(SRC_DIR.rglob(f"{stem}{ext}"))

    if len(candidates) > 0:
        return sorted(candidates)[0]

    return None

def shape_to_bbox(shape: dict, img_w: int, img_h: int):
    """
    把 shape 轉成 bbox: (x1, y1, x2, y2)
    目前以 points 的 min/max 做包圍框，所以 rectangle / polygon 都能吃。
    """
    pts = shape.get("points", [])
    if not pts or len(pts) < 2:
        return None

    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]

    x1 = int(np.floor(min(xs)))
    y1 = int(np.floor(min(ys)))
    x2 = int(np.ceil(max(xs)))
    y2 = int(np.ceil(max(ys)))

    # clamp 到影像範圍內
    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w))
    y2 = max(0, min(y2, img_h))

    if x2 <= x1 or y2 <= y1:
        return None

    return x1, y1, x2, y2

def safe_name(s: str) -> str:
    """
    檔名安全化
    """
    keep = []
    for ch in s:
        if ch.isalnum() or ch in ["_", "-", "."]:
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)

# =========================
# 2) 主流程
# =========================
json_files = sorted(SRC_DIR.rglob("*.json"))

if not json_files:
    print(f"[ERROR] 找不到 json: {SRC_DIR}")

grand_total_raw_poly = 0          # 全部 _poly shape 原始總數
grand_total_unique_poly = 0       # 全部唯一 _poly label 總數
grand_total_duplicate_extra = 0   # 全部重複多出而被略過的數量
grand_total_saved = 0             # 全部實際切出的總數
processed_maps = 0
maps_with_dup = 0

print("===== legend crop report (dedup by label per map) =====")

for json_path in json_files:
    with open(json_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    img_path = find_image_for_json(json_path, meta)
    if img_path is None:
        print(f"[SKIP] 找不到對應圖片: {json_path}")
        continue

    # 用 cv2 讀圖，保留原始通道
    img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"[SKIP] 讀不到圖片: {img_path}")
        continue

    h, w = img.shape[:2]
    map_name = img_path.stem   # 例如 CO_DenverW
    out_dir = DST_ROOT / map_name

    # 先清掉舊資料夾，避免重跑後殘留舊檔
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    shapes = meta.get("shapes", [])

    # 先統計這張圖所有 _poly label
    poly_labels_all = []
    for shape in shapes:
        label = str(shape.get("label", "")).strip()
        if label.endswith("_poly"):
            poly_labels_all.append(label)

    label_counter = Counter(poly_labels_all)
    raw_poly_count = len(poly_labels_all)
    unique_poly_count = len(label_counter)
    duplicate_extra_count = raw_poly_count - unique_poly_count

    if duplicate_extra_count > 0:
        maps_with_dup += 1

    # 真正切圖：同一張圖內，同名 label 只保留第一次
    seen_labels = set()
    saved_count = 0
    skipped_duplicate_count = 0
    skipped_invalid_bbox_count = 0
    skipped_empty_crop_count = 0
    skipped_write_fail_count = 0

    for shape in shapes:
        label = str(shape.get("label", "")).strip()

        # 只切 poly 類
        if not label.endswith("_poly"):
            continue

        # 同一張圖內，同名 label 不重複切
        if label in seen_labels:
            skipped_duplicate_count += 1
            continue

        bbox = shape_to_bbox(shape, w, h)
        if bbox is None:
            skipped_invalid_bbox_count += 1
            continue

        x1, y1, x2, y2 = bbox
        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            skipped_empty_crop_count += 1
            continue

        out_name = f"{saved_count + 1:03d}_{safe_name(label)}.png"
        out_path = out_dir / out_name

        ok = cv2.imwrite(str(out_path), crop)
        if not ok:
            skipped_write_fail_count += 1
            continue

        seen_labels.add(label)
        saved_count += 1

    print(f"\n[{map_name}]")
    print(f"  json = {json_path}")
    print(f"  _poly 原始總數      = {raw_poly_count}")
    print(f"  _poly 唯一 label 數 = {unique_poly_count}")
    print(f"  重複多出數量        = {duplicate_extra_count}")
    print(f"  實際切出張數        = {saved_count}")

    if duplicate_extra_count > 0:
        print(f"  重複 label 明細：")
        for label, n in sorted(label_counter.items()):
            if n > 1:
                print(f"    - {label}: {n} 次")

    if skipped_invalid_bbox_count > 0:
        print(f"  [INFO] bbox 無效略過 = {skipped_invalid_bbox_count}")
    if skipped_empty_crop_count > 0:
        print(f"  [INFO] 空 crop 略過  = {skipped_empty_crop_count}")
    if skipped_write_fail_count > 0:
        print(f"  [INFO] 寫檔失敗略過 = {skipped_write_fail_count}")

    grand_total_raw_poly += raw_poly_count
    grand_total_unique_poly += unique_poly_count
    grand_total_duplicate_extra += duplicate_extra_count
    grand_total_saved += saved_count
    processed_maps += 1

print("\n" + "-" * 60)
print("===== overall report =====")
print(f"共處理地圖數            = {processed_maps}")
print(f"有重複 _poly 的地圖數   = {maps_with_dup}")
print(f"全部 _poly 原始總數     = {grand_total_raw_poly}")
print(f"全部 _poly 唯一 label 數= {grand_total_unique_poly}")
print(f"全部重複略過數量        = {grand_total_duplicate_extra}")
print(f"最後總共切出 legend 圖  = {grand_total_saved}")
print(f"輸出資料夾              = {DST_ROOT}")

===== legend crop report (dedup by label per map) =====

[CO_DenverW]
  json = /data/ch21908234/work/SOTA_DATA/AI4CMA_evaluation_block/_poly_eq_2/eval_data_perfomer/CO_DenverW.json
  _poly 原始總數      = 101
  _poly 唯一 label 數 = 101
  重複多出數量        = 0
  實際切出張數        = 101

[MT_Havre]
  json = /data/ch21908234/work/SOTA_DATA/AI4CMA_evaluation_block/_poly_eq_2/eval_data_perfomer/MT_Havre.json
  _poly 原始總數      = 52
  _poly 唯一 label 數 = 26
  重複多出數量        = 26
  實際切出張數        = 26
  重複 label 明細：
    - R_poly: 3 次
    - afj_poly: 2 次
    - al_poly: 2 次
    - alt_poly: 2 次
    - aqa_poly: 2 次
    - asr_poly: 2 次
    - caa_poly: 2 次
    - cad_poly: 2 次
    - cax_poly: 2 次
    - cly_poly: 2 次
    - crg_poly: 2 次
    - gg_poly: 2 次
    - jea_poly: 2 次
    - ke_poly: 2 次
    - kg_poly: 2 次
    - lca_poly: 2 次
    - lse_poly: 2 次
    - lu_poly: 2 次
    - lx_poly: 2 次
    - pgd_poly: 2 次
    - tks_e_poly: 2 次
    - tks_poly: 2 次
    - tlx_e_poly: 2 次
    - tlx_poly: 2 次
    - tlx_s_poly: 2 次

[MT_

In [2]:
# ===== 掃資料夾內所有 json，逐張統計重複的 _poly label =====
from pathlib import Path
import json
from collections import Counter

# =========================
# 0) 只改這裡
# =========================
JSON_ROOT = Path("/data/ch21908234/work/SOTA_DATA/AI4CMA_evaluation_block/_poly_eq_2/eval_data_perfomer")

# =========================
# 1) 找所有 json
# =========================
json_files = sorted(JSON_ROOT.rglob("*.json"))

if not json_files:
    print(f"[ERROR] 找不到 json: {JSON_ROOT}")

# =========================
# 2) 逐張統計
# =========================
maps_with_dup = 0
total_maps = 0

print("===== duplicate _poly label report (per map) =====")

for json_path in json_files:
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
    except Exception as e:
        print(f"\n[SKIP] 讀檔失敗: {json_path}\n  reason: {e}")
        continue

    shapes = meta.get("shapes", [])
    poly_labels = []

    for shp in shapes:
        label = str(shp.get("label", "")).strip()
        if label.endswith("_poly"):
            poly_labels.append(label)

    cnt = Counter(poly_labels)
    dup = {k: v for k, v in sorted(cnt.items()) if v > 1}

    total_maps += 1
    map_name = json_path.stem  # 例如 CO_DenverW / MT_Havre

    print(f"\n[{map_name}]")
    print(f"  json = {json_path}")
    print(f"  _poly 總數 = {len(poly_labels)}")
    print(f"  _poly 唯一 label 數 = {len(cnt)}")

    if dup:
        maps_with_dup += 1
        print(f"  重複 label 數 = {len(dup)}")
        for label, n in dup.items():
            print(f"    - {label}: {n} 次")
    else:
        print("  無重複 _poly label")

print("\n" + "=" * 60)
print(f"共掃描 {total_maps} 張地圖(json)")
print(f"其中有重複 _poly label 的地圖數 = {maps_with_dup}")

===== duplicate _poly label report (per map) =====

[CO_DenverW]
  json = /data/ch21908234/work/SOTA_DATA/AI4CMA_evaluation_block/_poly_eq_2/eval_data_perfomer/CO_DenverW.json
  _poly 總數 = 101
  _poly 唯一 label 數 = 101
  無重複 _poly label

[MT_Havre]
  json = /data/ch21908234/work/SOTA_DATA/AI4CMA_evaluation_block/_poly_eq_2/eval_data_perfomer/MT_Havre.json
  _poly 總數 = 52
  _poly 唯一 label 數 = 26
  重複 label 數 = 25
    - R_poly: 3 次
    - afj_poly: 2 次
    - al_poly: 2 次
    - alt_poly: 2 次
    - aqa_poly: 2 次
    - asr_poly: 2 次
    - caa_poly: 2 次
    - cad_poly: 2 次
    - cax_poly: 2 次
    - cly_poly: 2 次
    - crg_poly: 2 次
    - gg_poly: 2 次
    - jea_poly: 2 次
    - ke_poly: 2 次
    - kg_poly: 2 次
    - lca_poly: 2 次
    - lse_poly: 2 次
    - lu_poly: 2 次
    - lx_poly: 2 次
    - pgd_poly: 2 次
    - tks_e_poly: 2 次
    - tks_poly: 2 次
    - tlx_e_poly: 2 次
    - tlx_poly: 2 次
    - tlx_s_poly: 2 次

[MT_RedRockLakes]
  json = /data/ch21908234/work/SOTA_DATA/AI4CMA_evaluation_block/_po